In [ ]:
%load_ext autoreload
%autoreload 2
    
import numpy as np
import pandas as pd
from glob import glob
import re
from pathlib import Path
import matplotlib.pyplot as plt
from collections import defaultdict
from scipy.signal import find_peaks, peak_prominences


import sys
sys.path.insert(1, '../')
from larfitter import retrieve_responses
from waffles.data_classes.WaveformSet import WaveformSet
from waffles.data_classes.EasyWaveformCreator import EasyWaveformCreator
from utils import PATH_XE_AVERAGES
from utils_deconv import DeconvFitParams, process_deconvfit
from utils import setup_response_template
from waffles.np02_utils.load_utils import ch_read_template
from DeconvFitterVDWrapper import DeconvFitterVDWrapper
from utils_monitoring import load_database, black_list_pmt
from waffles.np02_utils.PlotUtils import matplotlib_plot_WaveformSetGrid
from waffles.np02_utils.AutoMap import ordered_modules_cathode, ordered_modules_membrane, ordered_modules_pmt, getModuleName, dict_endpoints_channels_list

import mplhep
mplhep.style.use(mplhep.style.ROOT)
plt.rcParams.update({
                     'font.size': 14,
                     'grid.linestyle': '--',
                     'axes.grid': True,
                     'figure.autolayout': True,
                     'figure.figsize': [14,6],
                    #  'font.family': 'sans-serif',
                    #  'font.sans-serif': ['Helvetica', 'Helvetica Neue', 'Nimbus Sans', 'Liberation Sans', 'Arial']
                     })

In [ ]:
path_with_analysis = Path(PATH_XE_AVERAGES)
analysis_name = 'average_waveforms_few_cuts-hvieirad'
# analysis_name = 'average_waveforms_pmt-hvieirad'
db = load_database(load_hv = False)
ppm_values = db['ppm'].unique()
print("PPM values:", ppm_values)

In [ ]:
dettypes = ["cathode", "membrane"]
# dettypes = ["pmt"]

def _weighted_mean_per_channel(run_list, ep, output_all, chinfo_all):
    result = {}
    for ch in dict_endpoints_channels_list[ep]:
        numerator = denominator = 0
        missing_runs = []
        for run in run_list:
            if ch not in output_all[run][ep]:
                missing_runs.append(run)
                continue
            wf = output_all[run][ep][ch]
            counts = chinfo_all[run][ep][ch]['counts']
            numerator += wf * counts
            denominator += counts
        if denominator > 0:
            result[ch] = {'raw': numerator / denominator }
        if missing_runs:
            print(f'Missing runs for {getModuleName(ep,ch)}: {missing_runs}')
    return result

def total_average_waveform(db, path_base: Path, ppm_values):

    output_all = {}
    chinfo_all = {}
    runs_per_ppm = defaultdict(lambda: defaultdict(list))

    for ppm in ppm_values:
        runs_for_ppm = db[(db['ppm'] == ppm) & (db['status'] == 'stable') & (db['HV']==154)]

        for run in runs_for_ppm['run'].tolist():

            for dettype in dettypes:

                if dettype == "cathode":
                    endpoint = 106
                elif dettype == "membrane":
                    endpoint = 107
                elif dettype == "pmt":
                    if run in black_list_pmt:
                        continue
                    endpoint = 110

                matches = list(path_base.glob(f"run0{run}_{dettype}_*"))
                if not matches:
                    print(f"Warning: run {run} ({dettype}) not found.")
                    continue

                run_folder = matches[0]

                output_run = {}
                chinfo_run = {}

                retrieve_responses(run_folder, output_run, chinfo_run)

                if endpoint not in output_run:
                    print(f"Warning: endpoint {endpoint} missing in run {run}")
                    continue

                if run not in output_all:
                    output_all[run] = {}
                    chinfo_all[run] = {}

                output_all[run][endpoint] = output_run[endpoint]
                chinfo_all[run][endpoint] = chinfo_run[endpoint]

                runs_per_ppm[ppm][endpoint].append(run)

    mean_output = {}
    for ppm, ep_dict in runs_per_ppm.items():
        mean_output[ppm] = {}

        for ep, run_list in ep_dict.items():
            if not run_list:
                continue
            channels = dict_endpoints_channels_list[ep]
            mean_output[ppm][ep] = _weighted_mean_per_channel(run_list, ep, output_all, chinfo_all)

    return mean_output




In [ ]:
mean_output = total_average_waveform(db, path_with_analysis/analysis_name, ppm_values)

In [ ]:
template = ch_read_template('templates_large_pulses_zero_subtract')
template.update(ch_read_template('templates_pmt'))
def generate_deconv(mean_output, template):
    dfitparams = dict(
        scinttype = 'lar',
        error = 0.05,
        filter_type = 'Gauss',
        cutoff_MHz = 2.5,
        dtime = 16,
    )
    for ppm, ep_ch_avgs in mean_output.items():
        for ep, ch_avgs in ep_ch_avgs.items():
            for ch, avgs in ch_avgs.items():
                wf = mean_output[ppm][ep][ch]['raw']
                cfitch = DeconvFitterVDWrapper(**dfitparams)
                process_deconvfit(ep, ch, response=wf, template=template[ep][ch], cfitch=cfitch, dofit=False)
                mean_output[ppm][ep][ch]['deconv'] = cfitch.deconvolved.copy()

generate_deconv(mean_output, template)

In [ ]:
wfset = EasyWaveformCreator.create_WaveformSet_dictEndpointCh(dict_endpoint_ch=dict_endpoints_channels_list)

In [ ]:
def find_max(ep, wfm):
    prominence = 0.5 if ep != 110 else 500
    peaks, _ = find_peaks(wfm, prominence=prominence)
    if len(peaks) == 0:
        x_max = 1200//16 if ep == 106 else 1600//16
        x_min = 800//16
        peaks = [np.argmax(wfm[x_min:x_max]) + x_min]

    return peaks[0], wfm[peaks[0]]
def plot_ratios_all_channels(mean_output, ppm_values):
    ppm_sorted = sorted(ppm_values)
    
    EXCLUDE_MODULES = ["P05"]
    
    labels = []
    ratio = {ppm: [] for ppm in ppm_sorted[1:]}

    for ep, chlist in dict_endpoints_channels_list.items():
        if ep not in mean_output[0]:
            continue
        for ch in chlist:
            
            if getModuleName(ep, ch) in EXCLUDE_MODULES:
                continue
            
            for ppm in ppm_sorted[1:]:
                if ppm not in mean_output or ep not in mean_output[ppm] or ch not in mean_output[ppm][ep]:
                    ratio[ppm].append(None)
                else:
                    _, max_v = find_max(ep, mean_output[ppm][ep][ch]['deconv'])
                    _, max_ref = find_max(ep, mean_output[0][ep][ch]['deconv'])

                    ratio[ppm].append(max_v/max_ref) 

            labels.append(f"{getModuleName(ep, ch)}: {ep}-{ch}")

            
    fig, ax = plt.subplots(figsize=(max(8, len(labels) * 0.6), 5))
   
    for ppm in ratio:
        x = [ i for i, r in enumerate(ratio[ppm]) if r ]
        y = [ r for r in ratio[ppm] if r ]
        ax.plot(x, y, marker='o', label=f"{ppm:.02f} ppm / 0.00 ppm")

    x = np.arange(len(labels))
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=12)
    ax.set_xlabel("Modules")
    ax.set_ylabel("Ratio")
    ax.set_title("Average WFs max amplitude ratio")
    ax.legend(fontsize=14)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    # plt.show()

In [ ]:
plot_ratios_all_channels(mean_output, ppm_values)
# plt.savefig("Average_max_ratio")

In [ ]:
maxpos = {}

def plot_average(wfs:WaveformSet, mean_output, ppm_values, dodeconv:bool = True, normalization:bool = False, logscale=True):
    ax = plt.gca()
    ch = wfs.channel
    ep = wfs.endpoint 

    prop_cycle_colors = [ '#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf',]
    
    ppm_colors = {
        ppm: prop_cycle_colors[i % len(prop_cycle_colors)]
        for i, ppm in enumerate(sorted(ppm_values))
    }

    for ppm in sorted(ppm_values):
        if ppm not in mean_output:
            continue
        if ep not in mean_output[ppm]:
            continue
        if ch not in mean_output[ppm][ep]:
            continue
        if ep not in template or ch not in template[ep]:
            continue
            
        color = ppm_colors[ppm]

        wftype = 'raw' if not dodeconv else 'deconv'
        wf = mean_output[ppm][ep][ch][wftype]
    
        n_ticks = len(wf)
        x_ns = np.arange(n_ticks) * 16


        argmax, max_v = find_max(ep, wf)
        if ppm == 0:
            maxpos[(ep, ch)] = argmax

        shift = maxpos[(ep,ch)] - argmax
        wf = np.roll(wf, shift)
        norm = max_v if normalization else 1.

        ax.plot(x_ns, wf/norm, label=f"{ppm} ppm", color=color)            
    
        ax.legend(fontsize=12, loc='upper right')
            
            
    # ax.set_title(f"{getModuleName(ep, ch)}: {ep}-{ch}")
    ax.set_xlabel("Time [ns]")
    ax.set_ylabel("Amplitude [a.u.]")
    # ax.set_xlim(950, 1500)
    # ax.set_ylim(1e1, None)
    if logscale:
        ax.set_yscale('log')

In [ ]:
def genfilename(plotparams, detector_type):
    flag_names = { "dodeconv": (f"deconvolved_{detector_type}", f"raw_{detector_type}"), "logscale": ("log", None), "normalization": ("norm", None), }
    filename = '_'.join(
        (labels[0] if plotparams.get(key) else labels[1])
        for key, labels in flag_names.items()
        if (plotparams.get(key) and labels[0]) or (not plotparams.get(key) and labels[1])
    )
    return filename

In [ ]:
dodeconv=True
normalization=True
logscale=True
plotparams = {"mean_output": mean_output, "ppm_values": ppm_values, "dodeconv": dodeconv, "normalization": normalization, 'logscale': logscale}
fig, axs = matplotlib_plot_WaveformSetGrid(
    wfset,
    detector=ordered_modules_cathode,
    plot_function=plot_average,
    func_params=plotparams,
    cols=4, 
    figsize=(32, 16) 
)
detector_type='cathode'
filename = genfilename(plotparams, detector_type)
plt.savefig(f"{filename}.png")


In [ ]:
fig, axs = matplotlib_plot_WaveformSetGrid(
    wfset,
    detector=ordered_modules_membrane,
    plot_function=plot_average,
    func_params=plotparams,
    cols=4, 
    figsize=(32, 16) 
)
detector_type='membrane'
# filename = genfilename(plotparams, detector_type)
# plt.savefig(f"{filename}.png")

In [ ]:
dodeconv=True
normalization=False
logscale=False
plotparams = {"mean_output": mean_output, "ppm_values": ppm_values, "dodeconv": dodeconv, "normalization": normalization, 'logscale': logscale}
fig, axs = matplotlib_plot_WaveformSetGrid(
    wfset,
    detector=ordered_modules_cathode,
    plot_function=plot_average,
    func_params=plotparams,
    cols=4, 
    figsize=(32, 16) 
)

detector_type='cathode'
filename = genfilename(plotparams, detector_type)

for ax in np.ravel(axs):
    ax.set_xlim(50,5000)

plt.savefig(f"{filename}.png")
# plt.savefig("deconvolved_membrane_log_norm_few.png")

In [ ]:
fig, axs = matplotlib_plot_WaveformSetGrid(
    wfset,
    detector=ordered_modules_membrane,
    plot_function=plot_average,
    func_params=plotparams,
    cols=4, 
    figsize=(32, 16)   
)

detector_type='membrane'
filename = genfilename(plotparams, detector_type)
for ax in np.ravel(axs):
    ax.set_xlim(50,5000)

plt.savefig(f"{filename}.png")
# plt.savefig("deconvolved_membrane_log_norm_few.png")

In [ ]:
from pathlib import Path
import os
ananame_pieces = analysis_name.split('-')
if len(ananame_pieces)==1:
    ananame_pieces.append(os.getlogin())
analysisname_stable = PATH_XE_AVERAGES / Path(ananame_pieces[0] + "_deconv_stable-" + ananame_pieces[1])
analysisname_stable.mkdir(exist_ok=True, parents=True)
analysisname_stable.chmod(0o775) # apparently it needs to be done after

for ppm, ep_ch_avgs in mean_output.items():
    for ep, ch_avgs in ep_ch_avgs.items():
        for ch, avgs in ch_avgs.items():
            wf = mean_output[ppm][ep][ch]['deconv']
            filename = f'ppm_{ppm:.2f}_ep_{ep}_ch_{ch}.txt'
            np.savetxt(analysisname_stable / filename, wf)


In [ ]:
dodeconv=True
normalization=True
logscale=True
plotparams = {"mean_output": mean_output, "ppm_values": ppm_values, "dodeconv": dodeconv, "normalization": normalization, 'logscale': logscale}
fig, axs = matplotlib_plot_WaveformSetGrid(
    wfset,
    detector=["M7(1)", "M8(1)"],
    plot_function=plot_average,
    func_params=plotparams,
    cols=1,
    figsize=(10, 7)   
)

detector_type='membrane'
filename = genfilename(plotparams, detector_type)
for ax in np.ravel(axs):
    # ax.set_xlim(50O,5000)
    ax.set_xlim(50,16e3)
    ax.set_ylim(1e-5, 2)

plt.savefig(f"{filename}_sample.png")
# plt.savefig("deconvolved_membrane_log_norm_few.png")